# Day 12 — Locks: the Lost-Update Race, Visualized

`x += 1` is load-add-store, not atomic. Two threads can read the same value and one
increment vanishes. Watch it happen, then watch a lock fix it.

In [ ]:
import threading

def run_counter(use_lock, threads_n=8, per_thread=50_000):
    value = [0]; lock = threading.Lock()
    def work():
        for _ in range(per_thread):
            if use_lock:
                with lock: value[0] += 1
            else:
                value[0] += 1
    ts = [threading.Thread(target=work) for _ in range(threads_n)]
    for t in ts: t.start()
    for t in ts: t.join()
    return value[0], threads_n * per_thread

got, expected = run_counter(use_lock=False)
print(f'NO lock:   got {got:,} of {expected:,}  (lost {expected - got:,} updates)')
got, expected = run_counter(use_lock=True)
print(f'WITH lock: got {got:,} of {expected:,}  (exact)')

## Lost updates across several unlocked runs

In [ ]:
import matplotlib.pyplot as plt

runs = [run_counter(use_lock=False) for _ in range(6)]
expected = runs[0][1]
losses = [expected - got for got, _ in runs]
plt.figure(figsize=(6, 3.5))
plt.bar(range(len(losses)), losses, color='#DD8452')
plt.axhline(0, color='#55A868', lw=2, label='with a lock: always 0')
plt.xlabel('unlocked run #'); plt.ylabel('updates lost to the race')
plt.title('Races are nondeterministic — and always wrong'); plt.legend(); plt.show()

## Takeaways

- Unprotected read-modify-write across threads silently loses updates.
- A lock makes the critical section mutually exclusive and the result exact.
- For waiting on a condition (buffer full/empty), use a condition variable and a `while` loop.

**Now build** SafeCounter and BoundedBuffer in `homework.py`, then `pytest -q`.